In [6]:
import sys
sys.path.append("..")

from utils import *

df = pd.read_parquet("../data/raw_data.parquet")

os.makedirs("../plots", exist_ok=True)

---
## Preprocessing
Defined two text preprocessing functions used for cleaning and preparing raw text data for diffrent tasks.

- clean_minimal(text) performs basic cleaning by removing HTML tags, URLs, and extra whitespace, keeping the text largely unchanged otherwise.
- clean_full(text) applies a more advanced NLP pipeline: it removes HTML and URLs, expands contractions, filters out special characters and numbers, converts text to lowercase, tokenizes it, removes stopwords (while preserving important negations like “not” or “never”), lemmatizes words to their base form, and removes very short tokens.

In [7]:
def clean_minimal(text):
    text = re.sub(r'<[^>]+>', ' ', text) # Remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # Remove URLs
    text = re.sub(r'\s+', ' ', text).strip() # Normalize whitespace
    return text

def clean_full(text):
    text = re.sub(r'<[^>]+>', ' ', text) # Remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # Remove URLs
    text = contractions.fix(text)  # Expand contractions
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # Remove special characters and numbers 
    text = text.lower() # Lowercase
    tokens = word_tokenize(text) # Tokenize
    stop_words = set(stopwords.words('english')) # Remove stopwords
    stop_words -= {'not', 'no', 'nor', 'never', "don't", "doesn't", "didn't",
              "won't", "wouldn't", "can't", "couldn't", "shouldn't",
              "isn't", "aren't", "wasn't", "weren't"}
    tokens = [word for word in tokens if word not in stop_words]
    lemmatizer = WordNetLemmatizer() # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    tokens = [word for word in tokens if len(word) > 1] # Remove short tokens (length < 2)
    return ' '.join(tokens)

In [8]:
df['text_raw'] = df['text'].apply(clean_minimal)
df['text_clean'] = df['text'].apply(clean_full)
    
df['text_length_original'] = df['text'].apply(len)
df['text_length_raw'] = df['text_raw'].apply(len)
df['text_length_clean'] = df['text_clean'].apply(len)
df['word_count_clean'] = df['text_clean'].apply(lambda x: len(x.split()))

In [9]:
print('\n---------- Example: ---------')
sample = df.iloc[10]
print(f"\nORIGINAL:\n{sample['text']}...")
print(f"\nTEXT_RAW:\n{sample['text_raw']}...")
print(f"\nTEXT_CLEAN:\n{sample['text_clean']}...")

print('\n---------- Statistics: ----------')
display(df[['text_length_original', 'text_length_raw',
            'text_length_clean', 'word_count_clean']].describe().round(2))

df.head()


---------- Example: ---------

ORIGINAL:
Lars von Trier's Europa is a worthy echo of The Third Man, about an American coming to post-World War II Europe and finds himself entangled in a dangerous mystery.<br /><br />Jean-Marc Barr plays Leopold Kessler, a German-American who refused to join the US Army during the war, arrives in Frankfurt as soon as the war is over to work with his uncle as a sleeping car conductor on the Zentropa Railway. What he doesn't know is the war is still secretly going on with an underground terrorist group called the Werewolves who target American allies. Leopold is strongly against taking any sides, but is drawn in and seduced by Katharina Hartmann (Barbara Sukowa), the femme fatale daughter of the owner of the railway company. Her father was a Nazi sympathizer, but is pardoned by the American Colonel Harris (Eddie Considine) because he can help get the German transportation system up and running again. The colonel soon enlists, or forces, Leopold to be a s

,text_length_original,text_length_raw,text_length_clean,word_count_clean
count,50000.00,50000.00,50000.00,50000.00
mean,1309.43,1286.76,812.17,120.83
std,989.73,972.37,627.25,90.98
min,32.00,32.00,17.00,3.00
25%,699.00,690.00,425.00,65.00
50%,970.00,954.00,597.00,90.00
75%,1590.25,1561.00,987.00,147.00
max,13704.00,13593.00,9114.00,1411.00


,review_id,split,rating,text,sentiment,sentiment_label,movie_id,text_raw,text_clean,text_length_original,text_length_raw,text_length_clean,word_count_clean
0,127,train,7,Zentropa has much in common with The Third Man...,pos,1,None,Zentropa has much in common with The Third Man...,zentropa much common third man another noir li...,728,705,479,69
1,126,train,10,Zentropa is the most original movie I've seen ...,pos,1,None,Zentropa is the most original movie I've seen ...,zentropa original movie seen year like unique ...,673,673,419,60
2,125,train,7,Lars Von Trier is never backward in trying out...,pos,1,None,Lars Von Trier is never backward in trying out...,lars von trier never backward trying new techn...,2187,2121,1379,201
3,124,train,10,*Contains spoilers due to me having to describ...,pos,1,None,*Contains spoilers due to me having to describ...,contains spoiler due describe film technique r...,2544,2478,1514,239
4,123,train,10,That was the first thing that sprang to mind a...,pos,1,None,That was the first thing that sprang to mind a...,first thing sprang mind watched closing credit...,3225,3201,2031,306


In [10]:
df.to_parquet("../data/preprocessed_data.parquet")
print("Preprocessed data saved in", os.listdir("../data"))

Preprocessed data saved in ['raw_data.parquet', 'preprocessed_data.parquet']
